# Step 12.1 — Autoregressive Constraint：General Self-Attention 为什么不能直接拿来做 Next-Token LM？

Step 11 已经完成 general single-head self-attention：

$$
O=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V.
$$

但这个 general self-attention 默认允许每个 Query position 与**所有** Key positions 交互，包括它右边还没有出现的 future tokens。

对于 autoregressive language model（自回归语言模型），这会破坏 next-token prediction 的任务定义。

本节只建立这个约束：当前位置可以使用自己与过去的上下文，但不能读取未来 positions。如何把这个规则真正编码成 causal mask，属于 Step 12.2。


## 1. Next-Token Prediction：当前位置 t 到底在预测谁？

先把索引说清楚。

假设完整 token sequence 是：

$$
x_0,x_1,\ldots,x_T.
$$

训练时常把它 shift 成：

```text
input : x_0, x_1, ..., x_(T-1)
target: x_1, x_2, ..., x_T
```

因此 input position `t` 的任务是预测：

$$
\boxed{x_{t+1}}.
$$

也就是说，position `t` 合法拥有的 prefix context 是：

$$
x_0,x_1,\ldots,x_t.
$$

这里的 **present/current token `x_t` 是允许使用的**，因为目标是下一个 token `x_(t+1)`。

所以不要把两个不同任务混起来：

```text
本课程这里：用 x_0...x_t 预测 x_(t+1)
不是：      用 x_0...x_(t-1) 预测 x_t
```

两种索引写法都能描述 next-token modeling，但必须保持 input/target shift 一致。


In [ ]:
tokens = ['<BOS>', 'I', 'love', 'coffee', '<EOS>']

inputs = tokens[:-1]
targets = tokens[1:]

print('input :', inputs)
print('target:', targets)

for t, (x_t, target) in enumerate(zip(inputs, targets)):
    visible_prefix = inputs[: t + 1]
    print(
        f't={t}: current={x_t!r}, '
        f'visible_prefix={visible_prefix}, '
        f'predict={target!r}'
    )


## 2. Autoregressive 到底是什么意思？把 Joint Probability 拆成 Prefix-Conditioned Factors

**Autoregressive（自回归）**的核心不是“模型一定是 RNN”，而是概率分解方式。

对 token sequence：

$$
x_0,x_1,\ldots,x_T,
$$

可以按顺序写成：

$$
\boxed{
p(x_0,x_1,\ldots,x_T)
=
\prod_{t=0}^{T}
p(x_t\mid x_{<t})
}
$$

其中：

$$
x_{<t}
=
(x_0,\ldots,x_{t-1}).
$$

如果用上一节的 shifted-training 索引来描述，那么 input position `t` 用 prefix：

$$
x_{\le t}
$$

去预测：

$$
x_{t+1}.
$$

两种写法表达的是同一件事：

> **预测某个 token 时，条件信息只能来自这个 token 之前已经存在的 prefix。**

Autoregressive constraint 是任务的因果/信息约束，不等于“必须使用 recurrent architecture”。Transformer 可以没有 recurrent state transition，但依然执行 autoregressive factorization。


## 3. 把这个约束翻译成 Self-Attention 的 Row / Column 语言

Step 11.3 已经规定：

```text
Attention score matrix rows    = Query positions i
Attention score matrix columns = Key positions j
```

如果 Query position `i` 负责用 `x_0...x_i` 形成 representation 并预测 `x_(i+1)`，那么它合法访问的 Keys 必须满足：

$$
\boxed{j\le i}.
$$

也就是：

```text
j < i  -> past position，允许
j = i  -> current position，允许
j > i  -> future position，不允许
```

例如 `T=4`：

```text
Query 0 can use Key positions: [0]
Query 1 can use Key positions: [0,1]
Query 2 can use Key positions: [0,1,2]
Query 3 can use Key positions: [0,1,2,3]
```

注意这里仍只是在定义 **visibility rule（可见性规则）**。

本节不构造 mask tensor，不引入 upper-triangle `-inf`，也不解释 masked softmax；这些属于 Step 12.2。


In [ ]:
T = 4

for query_position in range(T):
    allowed_key_positions = list(range(query_position + 1))
    future_key_positions = list(range(query_position + 1, T))

    print(
        f'query={query_position}: '
        f'allowed={allowed_key_positions}, '
        f'future(disallowed)={future_key_positions}'
    )


## 4. Future Leakage 为什么等于 Cheating？

继续使用：

```text
input : <BOS>  I      love    coffee
target: I      love   coffee  <EOS>
```

看 input position `t=1`：

```text
current input = 'I'
target        = 'love'
```

合法信息只有：

```text
<BOS>, I
```

但是 general self-attention 如果允许 Query position 1 读取 future Key/Value position 2，它就会看到：

```text
input position 2 = 'love'
```

而 `'love'` 正好就是 position 1 要预测的 target。

这叫 **future leakage（未来信息泄漏）**。

模型可能学到一种作弊路径：

```text
我要预测 target token
↓
直接从右边 future position 读到 target 本身或与它强相关的信息
↓
训练 loss 很低
↓
但真实生成时 future token 根本不存在
```

因此训练时如果发生 future leakage，training condition 与 inference condition 不一致，得到的低 loss 也不能代表真正的 autoregressive prediction 能力。


## 5. 为什么训练时特别容易泄漏？因为整条 Target-Shifted Input 已经同时放在 Tensor 里

训练为了效率，会一次把很多 positions 放进同一个 tensor：

$$
X\in\mathbb R^{B\times T\times C}.
$$

所以从纯 tensor 角度看，position `i` 右边的 representations 已经存在于这一轮 forward pass 中。

General self-attention 又天然会构造：

$$
T\times T
$$

的 all-pairs Query–Key interactions。

因此如果我们什么都不限制，矩阵运算不会自动理解：

> “右边这些 token 虽然在训练 tensor 里，但对当前 prediction 来说属于未来，不能读取。”

Autoregressive LM 必须额外把这个任务规则注入 attention computation。

### Training 与 generation 的差别

Training：

```text
为了并行效率，一次放入整段 shifted input
但每个 position 必须假装自己只知道 prefix
```

Inference / generation：

```text
未来 token 本来就还没生成
所以天然不可用
```

这就是 causal constraint 的必要性：

> **让并行训练时每个 position 的信息条件，与真正逐 token 生成时保持一致。**


## 6. Step 12.1 到这里真正建立了什么？

从 Step 11 的 general self-attention：

```text
每个 Query position
可以读取所有 Key/Value positions
```

现在加入 autoregressive task requirement：

```text
Query position i
只能读取 j <= i 的 Key/Value positions
```

因此问题从：

> “Self-Attention 怎样让 positions 互相访问？”

进一步变成：

> “怎样在保持矩阵并行计算的同时，禁止每个 Query 看到未来？”

这个工程/数学机制就是下一节的 **causal mask**。

### Step 12.1 材料边界

- shifted next-token training 中，input position `t` 用 `x_0...x_t` 预测 `x_(t+1)`；current token `x_t` 属于合法上下文。
- autoregressive factorization 的本质是预测 token 时只能 condition on its prefix，不等于必须使用 RNN。
- 对 Self-Attention score matrix，row `i` 是 Query i、column `j` 是 Key j；autoregressive visibility 要求 `j<=i`。
- `j>i` 是 future position，当前 Query 不允许读取。
- future leakage 会让训练位置读到尚未生成的答案信息，造成 cheating，并破坏 training/inference 条件一致性。
- 训练可以一次并行放入整段 shifted input，但每个 position 仍必须遵守自己的 prefix visibility。
- 本轮只准备 Step 12.1；不进入 Step 12.2，不构造 causal mask，不使用 `-inf` 或 masked softmax。
